In [ ]:
from pathlib import Path
import re
import shutil
import subprocess
from typing import Any

import nemo.collections.asr as nemo_asr
import textgrid as tg


PROJECT_ROOT = Path.cwd().parent
AUDIO_PATH = PROJECT_ROOT / "data" / "allie" / "noise_removed.wav"
ASR_MODEL_PATH = PROJECT_ROOT / "models" / "parakeet-tdt-0.6b-v3.nemo"
WORK_DIR = PROJECT_ROOT / "data" / "allie" / "alignment_work"
ASR_TEXTGRID_PATH = WORK_DIR / "parakeet.TextGrid"
MFA_CORPUS_DIR = WORK_DIR / "mfa_corpus"
MFA_OUTPUT_DIR = WORK_DIR / "mfa_output"
FINAL_TEXTGRID_PATH = PROJECT_ROOT / "data" / "allie" / "noise_removed_aligned.TextGrid"

# MFA must be installed and these names must exist in its model database.
MFA_DICTIONARY = "english_us_arpa"
MFA_ACOUSTIC_MODEL = "english_us_arpa"
MFA_COMMAND = "mfa"

# download the ASR model if it doesn't exist
if not ASR_MODEL_PATH.exists():
    print("Downloading ASR model...")
    nemo_asr.load_model_from_url("https://api.ngc.nvidia.com/v2/models/nvidia/nemo/parakeet-tdt-0.6b-v3/versions/0.6.0/files/parakeet-tdt-0.6b-v3.nemo", ASR_MODEL_PATH)
else:
    print("ASR model already exists, skipping download.")

WORK_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
asr_model: Any = nemo_asr.models.ASRModel.restore_from(
    restore_path=str(ASR_MODEL_PATH),
    map_location="cpu",
)
asr_model = asr_model.to(device).eval()
print(f"Loaded Parakeet on {device}: {ASR_MODEL_PATH.name}")

In [ ]:
output = asr_model.transcribe(
    [str(AUDIO_PATH)],
    use_lhotse=False,
    num_workers=0,
    batch_size=1,
    timestamps=True,
)
asr_result = output[0]
segments = asr_result.timestamp["segment"]
words = asr_result.timestamp["word"]
print(f"Parakeet returned {len(segments)} segments and {len(words)} words")

In [ ]:
def _clean_label(value: Any) -> str:
    """Make ASR labels safe for TextGrid and MFA text input."""
    return re.sub(r"\s+", " ", str(value).replace("\n", " ")).strip()


def _write_asr_textgrid(
    segments: list[dict[str, Any]],
    words: list[dict[str, Any]],
    destination: Path,
) -> None:
    if not segments or not words:
        raise ValueError("Parakeet returned no segments or words")

    max_time = max(float(segments[-1]["end"]), float(words[-1]["end"]))
    grid = tg.TextGrid(minTime=0, maxTime=max_time)
    sentence_tier = tg.IntervalTier(name="sentence", minTime=0, maxTime=max_time)
    word_tier = tg.IntervalTier(name="word_asr", minTime=0, maxTime=max_time)

    for item in segments:
        sentence_tier.add(float(item["start"]), float(item["end"]), _clean_label(item["segment"]))
    for item in words:
        word_tier.add(float(item["start"]), float(item["end"]), _clean_label(item["word"]))

    grid.append(sentence_tier)
    grid.append(word_tier)
    grid.write(str(destination))


def _write_mfa_corpus(
    audio_path: Path,
    segments: list[dict[str, Any]],
    corpus_dir: Path,
) -> Path:
    """Create the one-file MFA corpus: audio plus a matching .lab transcript."""
    if corpus_dir.exists():
        shutil.rmtree(corpus_dir)
    corpus_dir.mkdir(parents=True)
    corpus_audio = corpus_dir / audio_path.name
    shutil.copy2(audio_path, corpus_audio)
    transcript = " ".join(_clean_label(item["segment"]) for item in segments)
    lab_path = corpus_audio.with_suffix(".lab")
    lab_path.write_text(transcript, encoding="utf-8")
    return lab_path


In [ ]:
_write_asr_textgrid(segments, words, ASR_TEXTGRID_PATH)
lab_path = _write_mfa_corpus(AUDIO_PATH, segments, MFA_CORPUS_DIR)
print(f"ASR TextGrid: {ASR_TEXTGRID_PATH}")
print(f"MFA transcript: {lab_path}")

In [ ]:
def _download_mfa_model(model_type: str, model_name: str) -> None:
    command = [MFA_COMMAND, "model", "download", model_type, model_name]
    result = subprocess.run(command, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if result.returncode != 0:
        raise RuntimeError(f"Could not install MFA {model_type} model {model_name}:\n{output}")
    print(output or f"MFA {model_type} model ready: {model_name}")


_download_mfa_model("dictionary", MFA_DICTIONARY)
_download_mfa_model("acoustic", MFA_ACOUSTIC_MODEL)

In [ ]:
def _run_mfa_alignment() -> Path:
    if MFA_OUTPUT_DIR.exists():
        shutil.rmtree(MFA_OUTPUT_DIR)
    MFA_OUTPUT_DIR.mkdir(parents=True)
    command = [
        MFA_COMMAND,
        "align",
        str(MFA_CORPUS_DIR),
        MFA_DICTIONARY,
        MFA_ACOUSTIC_MODEL,
        str(MFA_OUTPUT_DIR),
        "--clean",
        "--overwrite",
        "--single_speaker",
    ]
    print("Running:", " ".join(command))
    result = subprocess.run(command, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        details = result.stderr.strip() or result.stdout.strip() or "MFA returned no diagnostic output."
        raise RuntimeError(f"MFA alignment failed with exit code {result.returncode}:\n{details}")
    aligned_grids = sorted(MFA_OUTPUT_DIR.glob("*.TextGrid"))
    if len(aligned_grids) != 1:
        raise FileNotFoundError(f"Expected one MFA TextGrid, found {aligned_grids}")
    return aligned_grids[0]


mfa_textgrid_path = _run_mfa_alignment()
print(f"MFA output: {mfa_textgrid_path}")


In [ ]:
def _find_tier(grid: tg.TextGrid, names: tuple[str, ...]) -> tg.IntervalTier:
    for name in names:
        try:
            tier = grid.getFirst(name)
            if isinstance(tier, tg.IntervalTier):
                return tier
        except ValueError:
            continue
    raise ValueError(f"Could not find any of tiers {names}; found {grid.getNames()}")


def _copy_tier(source: tg.IntervalTier, name: str, min_time: float, max_time: float) -> tg.IntervalTier:
    copied = tg.IntervalTier(name=name, minTime=min_time, maxTime=max_time)
    for interval in source:
        copied.add(interval.minTime, interval.maxTime, _clean_label(interval.mark))
    return copied


asr_grid = tg.TextGrid.fromFile(str(ASR_TEXTGRID_PATH))
mfa_grid = tg.TextGrid.fromFile(str(mfa_textgrid_path))
max_time = max(float(asr_grid.maxTime), float(mfa_grid.maxTime))
final_grid = tg.TextGrid(minTime=0, maxTime=max_time)
final_grid.append(_copy_tier(_find_tier(asr_grid, ("sentence",)), "sentence", 0, max_time))
final_grid.append(_copy_tier(_find_tier(mfa_grid, ("words", "word")), "word", 0, max_time))
final_grid.append(_copy_tier(_find_tier(mfa_grid, ("phones", "phone")), "phoneme", 0, max_time))
final_grid.write(str(FINAL_TEXTGRID_PATH))
print(f"Final TextGrid: {FINAL_TEXTGRID_PATH}")
